In [1]:
%pip install -r ../requirements.txt


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\User\AppData\Local\Temp\ipykernel_22048\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
C:\Users\User\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def process_all_pdfs(pdf_directory):
    
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"\nfound {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
            print(f"✔️ Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f" ❌ Error: {e}")
        
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")


found 4 PDF files to process

Processing: datapot.vn-Practical-Statistics-for-Data-Scientists.pdf
✔️ Loaded 363 pages

Processing: Hands-On-Machine-Learning-with-Scikit-Learn-and-TensorFlow.pdf
✔️ Loaded 510 pages

Processing: thinker, tailor, soldier, spy.pdf
✔️ Loaded 445 pages

Processing: toaz.info-the-unending-game-a-former-rampaw-chiefs-insights-into-espionage-pdfdrive--pr_7b59117b63eb4a3f658d641a204f9b9e.pdf
✔️ Loaded 138 pages

Total documents loaded: 1456


In [11]:
def split_documents(documents,chunk_size=1000, chunk_overlap=200):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents info {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"\nContent: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [12]:
chunks = split_documents(all_pdf_documents)

Split 1456 documents info 3576 chunks

Example chunk:

Content: Peter Bruce, Andrew Bruce  
& Peter Gedeck
Second  
Edition
Practical
Statistics 
 for Data Scientists
50+ Essential Concepts Using R and Python...
Metadata: {'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2020-04-09T23:57:22+00:00', 'author': 'Peter Bruce;Andrew Bruce;Peter  Gedeck;', 'moddate': '2020-04-13T11:40:43+05:30', 'title': 'Practical Statistics for Data Scientists', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': '..\\data\\pdf\\datapot.vn-Practical-Statistics-for-Data-Scientists.pdf', 'total_pages': 363, 'page': 0, 'page_label': 'Cover', 'source_file': 'datapot.vn-Practical-Statistics-for-Data-Scientists.pdf', 'file_type': 'pdf'}


In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
from sentence_transformers.sentence_transformer import SentenceTransformer as SentenceTransformerModel


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformerModel(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7921.04it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\User\AppData\Local\Temp\ipykernel_22048\2824796383.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [16]:
from typing import List, Any
import numpy as np

In [17]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
    def retrieve(self, query_embedding, top_k=3):
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved_docs = []

        for i in range(len(results["documents"][0])):
            retrieved_docs.append({
                "content": results["documents"][0][i],
                "metadata": results["metadatas"][0][i]
            })

        return retrieved_docs

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [ ]:
valid_chunks = [
    doc for doc in chunks
    if doc.page_content is not None and str(doc.page_content).strip()
]

for doc in valid_chunks:
    doc.page_content = str(doc.page_content).encode(
        "utf-8", errors="replace"
    ).decode("utf-8")

print("Valid chunks:", len(valid_chunks))   # expect about 3576

if vectorstore.collection.count() == 0:
    texts = [doc.page_content for doc in valid_chunks]
    embeddings = embedding_manager.generate_embeddings(texts)
    vectorstore.add_documents(valid_chunks, embeddings)
else:
    print("Collection already populated, skipping ingest")

print("Docs in collection:", vectorstore.collection.count())

Generating embeddings for 3576 texts...


Batches: 100%|██████████| 112/112 [00:19<00:00,  5.85it/s]


Generated embeddings with shape: (3576, 384)
Adding 3576 documents to vector store...
Successfully added 3576 documents to vector store
Total documents in collection: 3576


In [21]:
def retrieve_documents(query, top_k=3):

    query_embedding = embedding_manager.model.encode(query)

    results = vectorstore.retrieve(
        query_embedding,
        top_k=top_k
    )

    return results

In [18]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model_name = os.getenv("model_name")
API_KEY = os.getenv("Token")

if not model_name:
    raise ValueError("model_name not found in .env")

if not API_KEY:
    raise ValueError("Token not found in .env")

print("Model:", model_name)

llm = ChatOpenAI(
    model=model_name,
    api_key=API_KEY,
    base_url="https://router.huggingface.co/v1",
    temperature=0.7,
    max_tokens=1024
)

Model: Qwen/Qwen3.8-27B


In [22]:
def rag_simple(query, llm, top_k=3):

    results = retrieve_documents(query, top_k)

    if not results:
        return "No relevant context found to answer the question."

    context = "\n\n".join(
        result["content"] for result in results
    )

    prompt = f"""
Use the following context to answer the question.
If the answer is not present in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

In [34]:
query = input("Enter your question: ")

answer = rag_simple(
    query,
    llm
)

print("\nAnswer:")
print(answer)


Answer:


Based on the provided documents, “snoden” appears to refer to **Snowden**, who is described as someone who made disclosures about intelligence agencies. The agencies describe him as **an agent of the Russians**, while Snowden says he went public because he was appalled by an intrusive breach of Americans’ constitutional right to privacy. Critics in the security establishment call him **a defector—an intelligence officer who takes up residence in a country whose spies are not friends**.
